# Compare Best mPFC Pipelines

This notebook compares the two strongest candidates from your pipeline sweep:

- `current_default`: low-pass -> pre-IRLS high-pass with mean restoration -> IRLS -> dF/F
- `double_exp_detrend`: low-pass -> double exponential detrend -> IRLS -> dF/F

Use the controls below to:

- choose a TDT trial from a root folder
- choose a comparison time window
- run both pipelines on the same trial
- inspect each stage side by side across the whole session and within your chosen window


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError(
        "This notebook requires ipywidgets. Install it in the active Jupyter environment first."
    ) from exc

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from compare_mpfc_pipelines import PIPELINES, run_pipeline

PIPELINE_NAMES = ["current_default", "double_exp_detrend"]
PIPELINE_CONFIGS = {cfg.name: cfg for cfg in PIPELINES if cfg.name in PIPELINE_NAMES}

if set(PIPELINE_CONFIGS) != set(PIPELINE_NAMES):
    missing = set(PIPELINE_NAMES) - set(PIPELINE_CONFIGS)
    raise RuntimeError(f"Missing expected pipeline configs: {sorted(missing)}")

plt.rcParams["figure.figsize"] = (16, 8)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


In [2]:
def discover_tdt_trials(root: str | Path) -> list[Path]:
    root = Path(root).expanduser()
    if not root.exists():
        return []

    block_dirs = []
    for path in root.rglob("*"):
        if not path.is_dir():
            continue
        names = {child.suffix.lower() for child in path.iterdir() if child.is_file()}
        if ".tsq" in names or ".tev" in names:
            block_dirs.append(path)

    return sorted(block_dirs)


def summarize_results(results: list[dict]) -> pd.DataFrame:
    rows = []
    for result in results:
        row = dict(result["metrics"])
        row["trial"] = Path(result["trial_path"]).name
        rows.append(row)
    columns = [
        "pipeline",
        "description",
        "trial",
        "da_fit_corr",
        "residual_mad",
        "residual_mean_abs",
        "dff_std",
        "zscore_std",
        "low_freq_fraction_lt_0p01Hz",
        "signal_fraction_0p01_to_0p5Hz",
        "transient_snr",
    ]
    return pd.DataFrame(rows)[columns]


def window_mask(timestamps: np.ndarray, start_time: float | None, end_time: float | None) -> np.ndarray:
    mask = np.ones_like(timestamps, dtype=bool)
    if start_time is not None:
        mask &= timestamps >= start_time
    if end_time is not None:
        mask &= timestamps <= end_time
    return mask


def plot_stage_grid(results: list[dict], start_time: float | None = None, end_time: float | None = None, title_prefix: str = ""):
    fig, axes = plt.subplots(len(results), 4, figsize=(22, 4.5 * len(results)), sharex=False)
    if len(results) == 1:
        axes = np.array([axes])

    for row_idx, result in enumerate(results):
        trial = result["trial"]
        cfg = result["config"]
        mask = window_mask(trial.timestamps, start_time, end_time)
        t = trial.timestamps[mask]
        ax0, ax1, ax2, ax3 = axes[row_idx]

        ax0.plot(t, result["raw_da"][mask], color="steelblue", linewidth=1.1, label="DA")
        ax0.plot(t, result["raw_isos"][mask], color="darkorange", linewidth=1.1, label="ISOS")
        ax0.set_title(f"{cfg.name}: raw")
        ax0.set_ylabel("Voltage (V)")
        if row_idx == 0:
            ax0.legend(loc="upper right")

        ax1.plot(t, result["pre_irls_da"][mask], color="steelblue", linewidth=1.1, label="DA")
        ax1.plot(t, result["pre_irls_isos"][mask], color="darkorange", linewidth=1.1, label="ISOS")
        ax1.set_title(f"{cfg.name}: pre-IRLS")
        if row_idx == 0:
            ax1.legend(loc="upper right")

        ax2.plot(t, trial.updated_DA[mask], color="steelblue", linewidth=1.1, label="DA")
        ax2.plot(t, trial.isosbestic_fitted[mask], color="darkorange", linewidth=1.1, label="Fitted ISOS")
        ax2.set_title(f"{cfg.name}: IRLS fit")
        if row_idx == 0:
            ax2.legend(loc="upper right")

        ax3.plot(t, trial.dFF[mask], color="forestgreen", linewidth=1.1, label="dF/F")
        ax3.plot(t, trial.zscore[mask], color="purple", linewidth=1.0, alpha=0.8, label="z-score")
        ax3.axhline(0, color="black", linewidth=0.8, alpha=0.4)
        ax3.set_title(f"{cfg.name}: normalized")
        if row_idx == 0:
            ax3.legend(loc="upper right")

        for ax in (ax0, ax1, ax2, ax3):
            ax.set_xlabel("Time (s)")

    heading = f"{title_prefix} Pipeline Stage Comparison".strip()
    fig.suptitle(heading, fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_normalized_overlay(results: list[dict], start_time: float | None = None, end_time: float | None = None):
    fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)
    colors = {
        "current_default": "teal",
        "double_exp_detrend": "crimson",
    }

    for result in results:
        trial = result["trial"]
        cfg = result["config"]
        mask = window_mask(trial.timestamps, start_time, end_time)
        t = trial.timestamps[mask]
        color = colors.get(cfg.name, None)
        axes[0].plot(t, trial.dFF[mask], linewidth=1.1, color=color, label=cfg.name)
        axes[1].plot(t, trial.zscore[mask], linewidth=1.1, color=color, label=cfg.name)

    axes[0].axhline(0, color="black", linewidth=0.8, alpha=0.4)
    axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.4)
    axes[0].set_title("dF/F overlay")
    axes[1].set_title("z-score overlay")
    axes[0].set_ylabel("dF/F")
    axes[1].set_ylabel("z-score")
    axes[1].set_xlabel("Time (s)")
    axes[0].legend(loc="upper right")
    axes[1].legend(loc="upper right")
    plt.tight_layout()
    plt.show()


def run_best_pipeline_comparison(
    trial_path: str | Path,
    da_stream: str = "_465A",
    isos_stream: str = "_405A",
    target_fs: float = 100.0,
    trim_start: float = 150.0,
    trim_end: float = 10.0,
    lowpass_cutoff: float = 3.0,
    highpass_cutoff: float = 0.001,
    irls_constant: float = 3.0,
    zscore_method: str = "standard",
):
    trial_path = Path(trial_path).expanduser().resolve()
    results = []
    for pipeline_name in PIPELINE_NAMES:
        config = PIPELINE_CONFIGS[pipeline_name]
        result = run_pipeline(
            trial_path=str(trial_path),
            da_stream=da_stream,
            isos_stream=isos_stream,
            config=config,
            target_fs=target_fs,
            trim_start=trim_start,
            trim_end=trim_end,
            lowpass_cutoff=lowpass_cutoff,
            highpass_cutoff=highpass_cutoff,
            irls_constant=irls_constant,
            zscore_method=zscore_method,
        )
        result["trial_path"] = str(trial_path)
        results.append(result)
    return results


In [3]:
trial_root = widgets.Text(
    value="",
    description="Trial root",
    layout=widgets.Layout(width="900px"),
    placeholder=r"Paste a folder that contains one or more TDT block folders",
)

refresh_button = widgets.Button(description="Refresh trials", button_style="info")
trial_dropdown = widgets.Dropdown(options=[], description="Trial", layout=widgets.Layout(width="900px"))

da_stream = widgets.Text(value="_465A", description="DA stream")
isos_stream = widgets.Text(value="_405A", description="ISOS stream")
target_fs = widgets.FloatText(value=100.0, description="Target fs")
trim_start = widgets.FloatText(value=150.0, description="Trim start")
trim_end = widgets.FloatText(value=10.0, description="Trim end")
lowpass_cutoff = widgets.FloatText(value=3.0, description="Low-pass")
highpass_cutoff = widgets.FloatText(value=0.001, description="High-pass")
irls_constant = widgets.FloatText(value=3.0, description="IRLS c")
zscore_method = widgets.Dropdown(options=["standard", "baseline", "modified"], value="standard", description="z-score")

window_start = widgets.FloatText(value=300.0, description="Window start")
window_end = widgets.FloatText(value=360.0, description="Window end")
run_button = widgets.Button(description="Run comparison", button_style="success")
output = widgets.Output()


def refresh_trials(_=None):
    root = trial_root.value.strip()
    if not root:
        trial_dropdown.options = []
        return

    trials = discover_tdt_trials(root)
    options = [(str(path.relative_to(root)), str(path)) for path in trials]
    trial_dropdown.options = options
    if options:
        trial_dropdown.value = options[0][1]


def run_and_plot(_=None):
    output.clear_output()
    with output:
        selected_trial = trial_dropdown.value
        if not selected_trial:
            print("Select a trial first.")
            return

        results = run_best_pipeline_comparison(
            trial_path=selected_trial,
            da_stream=da_stream.value.strip(),
            isos_stream=isos_stream.value.strip(),
            target_fs=target_fs.value,
            trim_start=trim_start.value,
            trim_end=trim_end.value,
            lowpass_cutoff=lowpass_cutoff.value,
            highpass_cutoff=highpass_cutoff.value,
            irls_constant=irls_constant.value,
            zscore_method=zscore_method.value,
        )

        display(Markdown(f"## Trial: `{Path(selected_trial).name}`"))
        display(summarize_results(results).round(4))

        plot_stage_grid(results, title_prefix="Whole session")
        plot_stage_grid(
            results,
            start_time=window_start.value,
            end_time=window_end.value,
            title_prefix=f"Window {window_start.value:g}-{window_end.value:g}s",
        )
        plot_normalized_overlay(results)
        plot_normalized_overlay(results, start_time=window_start.value, end_time=window_end.value)


refresh_button.on_click(refresh_trials)
run_button.on_click(run_and_plot)

controls = widgets.VBox([
    widgets.HBox([trial_root, refresh_button]),
    trial_dropdown,
    widgets.HBox([da_stream, isos_stream, target_fs, zscore_method]),
    widgets.HBox([trim_start, trim_end, lowpass_cutoff, highpass_cutoff, irls_constant]),
    widgets.HBox([window_start, window_end, run_button]),
])

display(controls, output)


Output()

### Notes

- `Trial root` should point to a folder that contains one or more TDT block folders somewhere beneath it.
- Click `Refresh trials` after changing the root path.
- The notebook shows both the full session and the selected window so you can inspect global drift and local event structure separately.
- If your baseline z-score mode needs baseline start/end times, keep using `standard` or extend the notebook with those fields before switching methods.
